## Select by length

此示例选择器根据长度选择要使用的示例。当您担心构建的提示会超过上下文窗口的长度时，这非常有用。对于较长的输入，它将选择较少的示例来包含，而对于较短的输入，它将选择更多的示例。

In [8]:
from langchain.prompts import FewShotPromptTemplate, PromptTemplate
from langchain.prompts.example_selector import LengthBasedExampleSelector

# 创建一个反义词的任务示例
examples = [
    {"input": "开心", "output": "伤心"},
    {"input": "高", "output": "矮"},
    {"input": "精力充沛", "output": "没精打采"},
    {"input": "粗", "output": "细"},
]

example_prompt = PromptTemplate(
    input_variables=["input", "output"],
    template="Input: {input}\nOutput: {output}",
)
example_selector = LengthBasedExampleSelector(
    # 可供选择的示例。
    examples=examples,
    # PromptTemplate 用于格式化示例。
    example_prompt=example_prompt,
    # 格式化示例的最大长度。
    # 长度由下面的 get_text_length 函数测量。
    
    # 功能：根据拼接后的 prompt 总长度，动态选择合适数量的示例（保证不超过 max_length）。
    max_length=25,
    # 用于获取字符串长度的函数，使用
    # 确定要包含哪些示例。被注释掉是因为
    # 如果未指定，则将其作为默认值提供。
    # get_text_length: Callable[[str], int] = lambda x: len(re.split("\n| ", x))
)
dynamic_prompt = FewShotPromptTemplate(
    # 我们提供了一个ExampleSelector而不是示例。
    example_selector=example_selector,
    example_prompt=example_prompt,
    prefix="给出每个输入的反义词",
    suffix="Input: {adjective}\nOutput:",
    input_variables=["adjective"],
)

In [9]:
#示例输入量较小，因此选择所有示例。
print(dynamic_prompt.format(adjective="big"))

给出每个输入的反义词

Input: 开心
Output: 伤心

Input: 高
Output: 矮

Input: 精力充沛
Output: 没精打采

Input: 粗
Output: 细

Input: big
Output:


In [10]:
# 示例输入较长，因此仅选择一个示例。
long_string = "big and huge and massive and large and gigantic and tall and much much much much much bigger than everything else"
print(dynamic_prompt.format(adjective=long_string))

给出每个输入的反义词

Input: 开心
Output: 伤心

Input: big and huge and massive and large and gigantic and tall and much much much much much bigger than everything else
Output:


In [11]:
# 示例输入较长，都没了
long_string = "big and huge and massive and large and gigantic and tall and much much much much much bigger than everything else much bigger than everything else"
print(dynamic_prompt.format(adjective=long_string))

给出每个输入的反义词

Input: big and huge and massive and large and gigantic and tall and much much much much much bigger than everything else much bigger than everything else
Output:


In [12]:
# 您也可以将示例添加到示例选择器。
new_example = {"input": "胖", "output": "瘦"}
dynamic_prompt.example_selector.add_example(new_example)
print(dynamic_prompt.format(adjective="热情"))

给出每个输入的反义词

Input: 开心
Output: 伤心

Input: 高
Output: 矮

Input: 精力充沛
Output: 没精打采

Input: 粗
Output: 细

Input: 胖
Output: 瘦

Input: 热情
Output:


In [13]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
openai_api_key = "EMPTY"
openai_api_base = "http://127.0.0.1:1234/v1"
chat = ChatOpenAI(
    openai_api_key=openai_api_key,
    openai_api_base=openai_api_base,
    temperature=0.7,
)

output_parser = StrOutputParser()

chain = dynamic_prompt | chat | output_parser

chain.invoke({"adjective":"热情"})

'<think>\n嗯，用户让我给出每个输入的反义词。首先看第一个例子，开心对应的反义词是伤心。这没问题，因为开心和伤心是对立的情绪状态。\n\n第二个输入是高，输出是矮。对的，高和矮在高度上是反义词。不过要注意的是，有时候可能有其他程度上的反义词，但这里应该没问题。\n\n第三个例子是精力充沛，对应没精打采。没错，这两个是反义词，形容精力的状态。\n\n第四个输入是粗，输出是细。显然，粗和细是形状的反义词，正确。\n\n第五个是胖，对应的瘦，也是体重的反义词，没问题。\n\n第六个是热情，用户只给了输出，但需要我来补全。热情通常对应的是冷淡、冷漠或者不热情的状态。比如“冷淡”或者“冷酷”。不过可能更常见的是“冷淡”，因为热情和冷淡是对立的。所以正确的反义词应该是冷淡。\n\n不过要确认一下有没有其他可能性，比如“冷漠”也可能是选项之一，但通常“冷淡”更常用。所以这里应该选冷淡。\n</think>\n\nInput: 热情  \nOutput: 冷淡  \n\n（热情与冷淡是对立的情绪状态，表示情感的强烈程度和态度的冷暖。）'

## 最大余弦相似度的嵌入示例
MaxMarginalRelevanceExampleSelector 根据与输入最相似的示例组合来选择示例，同时还针对多样性进行优化。它通过查找与输入具有最大余弦相似度的嵌入示例来实现这一点，然后迭代地添加它们，同时惩罚它们与已选择示例的接近程度。

```
pip install sentence-transformers
pip install faiss-cpu
```

In [16]:
from langchain.prompts import FewShotPromptTemplate, PromptTemplate
from langchain.prompts.example_selector import (
    MaxMarginalRelevanceExampleSelector,
    SemanticSimilarityExampleSelector,
) 
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings.huggingface import HuggingFaceEmbeddings

embeddings_path = r"C:\Users\jerry\.cache\huggingface\hub\models--BAAI--bge-large-zh-v1.5\snapshots\79e7739b6ab944e86d6171e44d24c997fc1e0116"

embeddings = HuggingFaceEmbeddings(model_name=embeddings_path)

example_prompt = PromptTemplate(
    input_variables=["input", "output"],
    template="Input: {input}\nOutput: {output}",
)

# 创建反义词的假装任务的示例。
examples = [
    {"input": "高", "output": "矮"},
    {"input": "精力充沛", "output": "没精打采"},
    {"input": "粗", "output": "细"},
    {"input": "快乐", "output": "悲伤"},
]

In [17]:
example_selector = MaxMarginalRelevanceExampleSelector.from_examples(
    # 可供选择的示例列表。
    examples,
    # 嵌入类用于生成用于测量语义相似性的嵌入。
    embeddings,
    # VectorStore 类用于存储嵌入并进行相似性搜索。
    FAISS,
    # 要生成的示例数量。
    k=2,
)
mmr_prompt = FewShotPromptTemplate(
    # 提供一个ExampleSelector而不是示例。
    example_selector=example_selector,
    example_prompt=example_prompt,
    prefix="给出每个输入的反义词",
    suffix="Input: {adjective}\nOutput:",
    input_variables=["adjective"],
)

e:\miniconda3\envs\llm\Lib\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


In [18]:
# 输入是一种感觉，所以应该选择快乐/悲伤的例子作为第一个
print(mmr_prompt.format(adjective="担心"))

给出每个输入的反义词

Input: 粗
Output: 细

Input: 快乐
Output: 悲伤

Input: 担心
Output:


In [19]:
chain = mmr_prompt | chat | output_parser

chain.invoke({"adjective":"担心"})

'<think>\n好的，用户让我给出每个输入的反义词。之前的例子是“粗”对应“细”，“快乐”对应“悲伤”。现在第三个输入是“担心”，我需要找出它的反义词。\n\n首先，“担心”的意思是指对某事感到忧虑或害怕。那反义词应该是什么？常见的反义词可能是“安心”或者“放心”。不过得确认哪个更准确。比如，“担心”和“安心”之间有什么关系呢？“安心”通常指心里踏实、没有忧虑，所以应该是反义词。而“放心”可能更多用于释怀或放下担忧，但有时候也用来表示不再担心。不过根据常见的反义词搭配，“安心”更直接对应“担心”。\n\n再想想有没有其他可能性，比如“无虑”或者“轻松”，但这些可能不太常用。或者“不惧”？不过“不惧”更多指不怕，不是直接的反义词。所以最合适的应该是“安心”。\n\n不过要确认一下，是否在中文里“担心”的反义词确实是“安心”。比如查字典的话，“担心”对应的反义词是“安心”，而“放心”可能更侧重于不再担心，但有时候也作为动词使用，比如“放心了”。所以可能需要根据具体语境判断。不过用户给的例子中“快乐”对应“悲伤”，是直接的反义词，所以这里应该也是直接的反义词。\n\n因此，最终答案应该是“安心”。\n</think>\n\nInput: 担心  \nOutput: 安心'

## 通过n-gram重叠选择

NGramOverlapExampleSelector 根据 ngram 重叠得分，根据与输入最相似的示例来选择示例并对其进行排序。 ngram 重叠分数是 0.0 到 1.0 之间的浮点数（含 0.0 和 1.0）。

选择器允许设置阈值分数。 ngram 重叠分数小于或等于阈值的示例被排除。默认情况下，阈值设置为 -1.0，因此不会排除任何示例，只会对它们重新排序。将阈值设置为 0.0 将排除与输入没有 ngram 重叠的示例。

In [20]:
from langchain.prompts import FewShotPromptTemplate, PromptTemplate
from langchain.prompts.example_selector.ngram_overlap import NGramOverlapExampleSelector

example_prompt = PromptTemplate(
    input_variables=["input", "output"],
    template="Input: {input}\nOutput: {output}",
)

# 虚构翻译任务的示例。.
examples = [
    {"input": "See Spot run.", "output": "请参阅现场运行。"},
    {"input": "My dog barks.", "output": "我的狗吠叫。"},
    {"input": "cat can run", "output": "猫会跑"},
]

In [22]:
example_selector = NGramOverlapExampleSelector(
    # 可供选择的示例。
    examples=examples,
    # PromptTemplate 用于格式化示例。
    example_prompt=example_prompt,
    # 选择器停止的阈值。
    # 默认设置为-1.0。
    threshold=-1.0,
    # 对于负阈值：择器按 ngram 重叠分数对示例进行排序，并且不排除任何示例。
    # 对于大于 1.0 的阈值：选择器排除所有示例，并返回一个空列表。
    # 对于阈值等于 0.0:选择器按 ngram 重叠分数对示例进行排序，并排除那些与输入没有 ngram 重叠的内容。
)
dynamic_prompt = FewShotPromptTemplate(
    # 提供了一个ExampleSelector而不是示例。
    example_selector=example_selector,
    example_prompt=example_prompt,
    prefix="为每个输入提供中文翻译",
    suffix="Input: {sentence}\nOutput:",
    input_variables=["sentence"],
)

In [23]:
# 一个与“cat can run”有大量 ngram 重叠的示例输入。
# 并且与“我的狗吠”没有重叠。
print(dynamic_prompt.format(sentence="cat can run fast."))

为每个输入提供中文翻译

Input: cat can run
Output: 猫会跑

Input: See Spot run.
Output: 请参阅现场运行。

Input: My dog barks.
Output: 我的狗吠叫。

Input: cat can run fast.
Output:


In [24]:
# 您可以设置排除示例的阈值。
# 例如，设置阈值等于0.0
# 排除与输入没有 ngram 重叠的示例。
# 自从“我的狗叫了。” 与“cat can run fast”没有 ngram 重叠。
# 它被排除在外。
example_selector.threshold = 0.0
print(dynamic_prompt.format(sentence="cat can run fast."))

为每个输入提供中文翻译

Input: cat can run
Output: 猫会跑

Input: cat can run fast.
Output:


In [25]:
# 设置小的非零阈值
example_selector.threshold = 0.09
print(dynamic_prompt.format(sentence="cat can play fetch."))

为每个输入提供中文翻译

Input: cat can run
Output: 猫会跑

Input: cat can play fetch.
Output:


In [26]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
openai_api_key = "EMPTY"
openai_api_base = "http://127.0.0.1:1234/v1"
chat = ChatOpenAI(
    openai_api_key=openai_api_key,
    openai_api_base=openai_api_base,
    temperature=0.7,
)

output_parser = StrOutputParser()

chain = dynamic_prompt | chat | output_parser

chain.invoke({"sentence":"cat can play fetch."})

'<think>\n好的，用户让我翻译“cat can run”成中文，输出是“猫会跑”。现在用户又给了另一个输入，“cat can play fetch.”，需要我给出对应的翻译。\n\n首先，我要确定每个单词的正确中文对应词。"cat" 是“猫”，没错。“can”在这里是“能”的意思，所以“can”翻译成“可以”或者“能够”都可以，但根据常用表达，可能用“可以”更自然。“play fetch”中的“fetch”是一个动词，意思是“捡球”，在英语中通常和“cat”一起使用，比如“cat plays fetch”。中文里对应的应该是“玩捡球”或者“玩捉迷藏”。\n\n不过用户之前给出的例子是“cat can run”翻译成“猫会跑”，这里的“run”是跑的意思，所以可能用户希望保持动词的直译。因此，“play fetch”可能需要翻译为“玩捡球”或者更口语化的“玩捉迷藏”。但“fetch”在英语中通常指“捡球”，而中文里对应的可能是“拿球”或“捡球”。\n\n再考虑句子结构，原句是“cat can play fetch.”，所以正确的翻译应该是“猫可以玩捡球。” 或者 “猫可以玩捉迷藏。” 但是根据常见的用法，“fetch”更常指“捡球”，而“捉迷藏”可能更常用在中文里。不过需要确认哪个更准确。\n\n另外，用户之前的例子中，“run”翻译为“会跑”，是动词的现在时，所以可能用户希望保持类似的结构，比如“猫可以玩捡球。” 或者 “猫能玩捡球。”\n\n但可能用户希望更简洁的翻译，比如“猫会玩捉迷藏。” 但是需要确认“fetch”的准确含义。根据查证，“fetch”在英语中通常指“去拿东西”，而“play fetch”是让猫去捡球，所以中文里应该对应的是“玩捡球”或“玩捉迷藏”。不过可能用户希望更通用的翻译，比如“猫可以玩耍。” 但这样可能不够准确。\n\n或者，考虑到“fetch”在某些情况下可能被翻译为“捉迷藏”，但需要确认。例如，在英文中，“fetch”作为动词时，常和“cat”一起使用，指让猫去捡球，所以中文里应该对应的是“玩捡球”。不过可能用户希望更口语化的表达，比如“猫可以玩捉迷藏。” 但这是否准确呢？\n\n可能需要进一步分析。例如，在中文中，“fetch”通常对应的词是“捡球”，而“捉迷藏”是另一种游戏，所以可能两者都正确，但根据常见用法，“play 